# Polymarket Simulation Engine — Interactive Demo

This notebook walks through each component of the simulation engine:
1. **Kyle's Lambda** — price impact estimation
2. **Hawkes Process** — order flow clustering
3. **VPIN** — adverse selection detection
4. **Almgren-Chriss** — optimal execution
5. **Avellaneda-Stoikov** — market making simulation
6. **Full simulation** — everything combined

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100
plt.style.use('dark_background')

## 1. Kyle's Lambda — Who Knows What You Don't

Lambda measures how much each dollar of order flow moves the price. High lambda = informed traders are active.

In [ ]:
from sim.kyle import estimate_kyle_lambda, simulate_kyle_market

# Generate synthetic market data with known lambda
data = simulate_kyle_market(n_trades=500, true_prob=0.65, seed=42)
result = estimate_kyle_lambda(data['prices'], data['volumes'], data['signs'])

print(f"True lambda:     0.0000015")
print(f"Estimated:       {result['lambda']:.7f}")
print(f"R-squared:       {result['r_squared']:.4f}")
print(f"Verdict:         {result['interpretation']}")

# Plot price evolution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(data['prices'], linewidth=0.8, color='cyan')
ax1.axhline(y=0.65, color='red', linestyle='--', alpha=0.5, label='True prob')
ax1.set_title("Price Evolution (Kyle Model)")
ax1.set_xlabel("Trade #")
ax1.set_ylabel("Price")
ax1.legend()

# Scatter: signed volume vs price change
signed_vol = data['volumes'] * data['signs']
price_changes = np.diff(data['prices'])
ax2.scatter(signed_vol, price_changes, alpha=0.3, s=10, color='cyan')
ax2.set_title(f"Price Impact Regression (lambda={result['lambda']:.7f})")
ax2.set_xlabel("Signed Volume")
ax2.set_ylabel("Price Change")

# Regression line
x_range = np.linspace(signed_vol.min(), signed_vol.max(), 100)
ax2.plot(x_range, result['lambda'] * x_range, color='red', linewidth=2)

plt.tight_layout()
plt.show()

## 2. Hawkes Process — When Orders Cluster

Order flow isn't uniform — it clusters. The branching ratio tells you what fraction of trades are caused by other trades vs. new information.

In [ ]:
from sim.hawkes import simulate_hawkes, fit_hawkes

# Simulate and fit
times = simulate_hawkes(mu=1.0, alpha=0.6, beta=2.0, T=3600, seed=123)
fitted = fit_hawkes(times, T=3600)

print(f"True:    mu=1.000, alpha=0.600, beta=2.000, branching=0.300")
print(f"Fitted:  mu={fitted['mu']:.3f}, alpha={fitted['alpha']:.3f}, "
      f"beta={fitted['beta']:.3f}, branching={fitted['branching_ratio']:.3f}")
print(f"\n{fitted['interpretation']}")

# Plot inter-arrival times to show clustering
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Event timeline (first 600 seconds)
mask = times < 600
ax1.eventplot([times[mask]], lineoffsets=0, linelengths=0.5, colors='cyan')
ax1.set_title("Order Arrivals (first 10 minutes)")
ax1.set_xlabel("Time (seconds)")
ax1.set_yticks([])

# Inter-arrival time histogram
iat = np.diff(times)
ax2.hist(iat, bins=50, color='cyan', alpha=0.7, edgecolor='white', linewidth=0.5)
ax2.axvline(x=1/fitted['avg_intensity'], color='red', linestyle='--', 
            label=f"Mean IAT = {1/fitted['avg_intensity']:.2f}s")
ax2.set_title("Inter-arrival Time Distribution")
ax2.set_xlabel("Seconds between trades")
ax2.set_ylabel("Count")
ax2.legend()

plt.tight_layout()
plt.show()

## 3. VPIN — Your Early Warning System

VPIN detects when one side of the market is trading much heavier than the other — a signal that informed traders have arrived.

In [ ]:
from sim.vpin import compute_vpin, rolling_vpin

rng = np.random.default_rng(42)

# Scenario: normal flow for 300 ticks, then informed buyer enters
buy_vol = np.concatenate([rng.exponential(100, 300), rng.exponential(300, 200)])
sell_vol = np.concatenate([rng.exponential(100, 300), rng.exponential(50, 200)])

vpin = rolling_vpin(buy_vol, sell_vol, window=50)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.bar(range(len(buy_vol)), buy_vol, color='green', alpha=0.6, label='Buy volume', width=1)
ax1.bar(range(len(sell_vol)), -sell_vol, color='red', alpha=0.6, label='Sell volume', width=1)
ax1.axvline(x=300, color='yellow', linestyle='--', label='Informed trader enters')
ax1.set_title("Order Flow")
ax1.legend()

ax2.plot(range(49, 49 + len(vpin)), vpin, color='cyan', linewidth=1.5)
ax2.axhline(y=0.40, color='yellow', linestyle='--', alpha=0.5, label='Widen spread')
ax2.axhline(y=0.65, color='orange', linestyle='--', alpha=0.5, label='Double spread')
ax2.axhline(y=0.80, color='red', linestyle='--', alpha=0.5, label='Pull quotes')
ax2.axvline(x=300, color='yellow', linestyle='--', alpha=0.3)
ax2.set_title("Rolling VPIN")
ax2.set_xlabel("Tick")
ax2.set_ylabel("VPIN")
ax2.legend()

plt.tight_layout()
plt.show()

## 4. Almgren-Chriss — Optimal Execution

How to split a large order to minimize the total cost of trading (impact + risk).

In [ ]:
from sim.execution import AlmgrenChrissParams, almgren_chriss_schedule

# Compare different risk aversion levels
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
risk_levels = [1e-7, 1e-6, 1e-5]
labels = ['Low risk aversion\n(spread evenly)', 'Moderate', 'High risk aversion\n(front-load)']

for ax, risk, label in zip(axes, risk_levels, labels):
    params = AlmgrenChrissParams(
        total_shares=10_000, T=4.0, N=8,
        sigma=0.02, eta=0.001, gamma=0.005,
        risk_aversion=risk
    )
    sched = almgren_chriss_schedule(params)
    
    ax.bar(sched['times'], sched['trade_sizes'], width=0.4, color='cyan', alpha=0.8)
    ax.set_title(f"{label}\nkappa={sched['kappa']:.3f}, cost=${sched['expected_cost']:.0f}")
    ax.set_xlabel("Hour")
    ax.set_ylabel("Order Size ($)")
    ax.set_ylim(0, max(sched['trade_sizes']) * 1.2)

plt.suptitle("Almgren-Chriss: $10,000 execution over 4 hours", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Print the moderate schedule
params = AlmgrenChrissParams(
    total_shares=10_000, T=4.0, N=8,
    sigma=0.02, eta=0.001, gamma=0.005, risk_aversion=1e-6
)
sched = almgren_chriss_schedule(params)
print(f"\nModerate schedule — expected cost: ${sched['expected_cost']:.2f} "
      f"({sched['implementation_shortfall']*100:.3f}%)")
print(f"{'Hour':>5}  {'Order $':>10}  {'Remaining $':>12}")
print("-" * 32)
for t, size, rem in zip(sched['times'], sched['trade_sizes'], sched['remaining']):
    print(f"{t:>5.1f}  {size:>10.0f}  {rem:>12.0f}")

## 5. Full Simulation — Market Making with All Defenses

Run the complete Avellaneda-Stoikov market maker with VPIN-triggered spread widening. Compare normal market vs. informed trader scenario.

In [ ]:
from sim.simulator import SimConfig, run_simulation, run_batch

# Single simulation — normal market
config_normal = SimConfig(true_prob=0.65, seed=42)
result_normal = run_simulation(config_normal)

# Single simulation — informed trader enters halfway through
config_informed = SimConfig(true_prob=0.65, informed_trader_active=True, 
                            informed_entry_step=500, seed=42)
result_informed = run_simulation(config_informed)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row, (result, label) in enumerate([(result_normal, "Normal Market"), 
                                         (result_informed, "With Informed Trader")]):
    # Price + quotes
    axes[row, 0].plot(result.prices, color='white', linewidth=0.8, label='Mid')
    axes[row, 0].plot(result.bid_history, color='green', linewidth=0.3, alpha=0.5, label='Bid')
    axes[row, 0].plot(result.ask_history, color='red', linewidth=0.3, alpha=0.5, label='Ask')
    axes[row, 0].axhline(y=0.65, color='yellow', linestyle='--', alpha=0.3)
    axes[row, 0].set_title(f"{label} — Price & Quotes")
    axes[row, 0].legend(fontsize=8)
    
    # Inventory
    axes[row, 1].plot(result.inventory_history, color='orange', linewidth=0.8)
    axes[row, 1].axhline(y=0, color='white', linestyle='--', alpha=0.3)
    axes[row, 1].set_title(f"{label} — Inventory")
    axes[row, 1].set_ylabel("$ exposure")
    
    # Spread
    axes[row, 2].plot(result.spreads, color='cyan', linewidth=0.5)
    axes[row, 2].set_title(f"{label} — Spread")
    axes[row, 2].set_ylabel("Bid-Ask Spread")

    if row == 0:
        for ax in axes[row]:
            ax.set_xlabel("")
    else:
        axes[row, 0].axvline(x=500, color='yellow', linestyle='--', alpha=0.5, label='Informed entry')

plt.tight_layout()
plt.show()

print(f"{'':>25} {'Normal':>12} {'Informed':>12}")
print("-" * 50)
print(f"{'Final P&L':>25} ${result_normal.final_pnl:>11.2f} ${result_informed.final_pnl:>11.2f}")
print(f"{'Final inventory':>25} ${result_normal.final_inventory:>11.0f} ${result_informed.final_inventory:>11.0f}")
print(f"{'Avg spread':>25} {result_normal.avg_spread:>12.4f} {result_informed.avg_spread:>12.4f}")
print(f"{'Total fills':>25} {result_normal.n_fills:>12d} {result_informed.n_fills:>12d}")
print(f"{'Kyle lambda':>25} {result_normal.kyle_lambda:>12.6f} {result_informed.kyle_lambda:>12.6f}")
print(f"{'Peak VPIN':>25} {result_normal.vpin_peak:>12.3f} {result_informed.vpin_peak:>12.3f}")

## 6. Batch Simulation — Statistical Confidence

Run 100+ simulations to get statistically meaningful P&L distributions.

In [ ]:
# Batch: 100 simulations each
stats_normal = run_batch(100, SimConfig(true_prob=0.65, seed=42))
stats_informed = run_batch(100, SimConfig(true_prob=0.65, informed_trader_active=True, seed=42))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

pnls_n = [r.final_pnl for r in stats_normal['results']]
pnls_i = [r.final_pnl for r in stats_informed['results']]

ax1.hist(pnls_n, bins=30, color='cyan', alpha=0.7, edgecolor='white', linewidth=0.5, label='Normal')
ax1.hist(pnls_i, bins=30, color='red', alpha=0.5, edgecolor='white', linewidth=0.5, label='Informed')
ax1.axvline(x=0, color='white', linestyle='--', alpha=0.5)
ax1.set_title("P&L Distribution (100 sims each)")
ax1.set_xlabel("Final P&L ($)")
ax1.legend()

# Cumulative P&L paths (first 10 sims)
for r in stats_normal['results'][:10]:
    if r.pnl_history:
        ax2.plot(r.pnl_history, color='cyan', alpha=0.3, linewidth=0.5)
for r in stats_informed['results'][:10]:
    if r.pnl_history:
        ax2.plot(r.pnl_history, color='red', alpha=0.3, linewidth=0.5)
ax2.axhline(y=0, color='white', linestyle='--', alpha=0.3)
ax2.set_title("P&L Paths (10 sample paths each)")
ax2.set_xlabel("Fill #")
ax2.set_ylabel("Mark-to-Market P&L ($)")

plt.tight_layout()
plt.show()

print(f"{'':>25} {'Normal':>12} {'Informed':>12}")
print("-" * 50)
print(f"{'Mean P&L':>25} ${stats_normal['mean_pnl']:>11.2f} ${stats_informed['mean_pnl']:>11.2f}")
print(f"{'Sharpe':>25} {stats_normal['sharpe']:>12.3f} {stats_informed['sharpe']:>12.3f}")
print(f"{'% profitable':>25} {stats_normal['pct_profitable']:>11.0f}% {stats_informed['pct_profitable']:>11.0f}%")
print(f"{'Avg spread':>25} {stats_normal['avg_spread']:>12.4f} {stats_informed['avg_spread']:>12.4f}")